<a href="https://colab.research.google.com/github/srfrat/energy-consumption-regression/blob/main/EIA_RECS_energy_consumption_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
library(googledrive)

# Authenticate your Google account
drive_auth(use_oob = TRUE, cache = FALSE)

# Download the file from your Drive into the Colab session
drive_download("recs2020_public_v7.csv", overwrite = TRUE)

# Read the downloaded file
df <- read.csv("recs2020_public_v7.csv")


Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=60a02011aea6145151b9d258c4d0c5c8&access_type=offline&prompt=consent



In [ ]:


data=read.csv("recs2020_public_v7.csv")

In [ ]:
head(data)

In [ ]:
attributes(data$BA_climate)

In [ ]:
dim(data)

In [ ]:
names(data)

In [ ]:
attributes(data)

In [15]:
named_cols <- c("TOTALDOL", "TOTSQFT_EN", "KOWNRENT", "NHSLDMEM", "TYPEHUQ", "FUELHEAT", "HDD65", "CDD65", "BA_climate", "NWEIGHT")
col_indices <- c(match(named_cols, names(data)), 552:611)
df = data[, col_indices]

In [ ]:
which(colnames(data)=="NWEIGHT60")

In [16]:
head(df)

,TOTALDOL,TOTSQFT_EN,KOWNRENT,NHSLDMEM,TYPEHUQ,FUELHEAT,HDD65,CDD65,BA_climate,NWEIGHT,⋯,NWEIGHT51,NWEIGHT52,NWEIGHT53,NWEIGHT54,NWEIGHT55,NWEIGHT56,NWEIGHT57,NWEIGHT58,NWEIGHT59,NWEIGHT60
,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2656.89,2100,1,2,2,1,3844,1679,Mixed-Dry,3284.104,⋯,3335.188,3240.132,3429.728,3294.084,3397.713,3292.610,0.000,3369.768,3358.163,3404.031
2,975.00,590,2,1,5,5,3766,1458,Mixed-Humid,9007.387,⋯,8955.288,9000.296,9290.375,9199.326,8958.782,9232.597,9140.427,9306.997,9061.782,8957.915
3,522.65,900,2,1,5,1,3819,1696,Mixed-Dry,5669.002,⋯,5635.561,5944.330,5683.500,5735.631,5674.564,5660.854,5917.193,5571.015,5887.092,5837.846
4,2061.77,2100,1,2,2,1,2614,1718,Mixed-Humid,5294.239,⋯,5447.619,5344.453,5437.803,5377.898,5357.470,5421.025,5365.230,5402.057,5402.772,5350.860
5,1463.04,800,2,2,5,1,4219,1363,Mixed-Humid,9935.465,⋯,10017.094,9954.142,10050.961,10018.844,10309.728,10142.763,10176.828,10043.143,10247.911,10110.301
6,2335.08,4520,2,4,2,1,901,3558,Hot-Humid,7249.540,⋯,7379.108,7498.391,7417.977,7230.113,7516.936,7370.323,7383.275,7507.097,7335.022,7594.642


In [17]:
names(df)=tolower(names(df))

# Logging the survey design


In [ ]:
install.packages("survey", repos = "https://cloud.r-project.org/")
library(survey)



In [18]:
design <- svrepdesign(
  data         = df,
  weights      = ~nweight,
  repweights   = "nweight[0-9]+",   # regex matches NWEIGHT1..NWEIGHT60, not NWEIGHT
  type         = "JK1",
  scale        = 59/60,
  rscales      = 1,
  mse          = TRUE,
  combined.weights = TRUE
)

#Variable recoding


In [34]:
design = update(design,
                main_heating_fuel = factor(fuelheat,
                                           levels = c(-2, 5, 1, 2, 3, 7, 99),
                                           labels = c('Not applicable',
                                                      'Electricity',
                                                      'Natural gas from underground pipes',
                                                      'Propane (bottled gas)',
                                                      'Fuel oil',
                                                      'Wood or pellets',
                                                      'Other')),
                type_of_household = factor(typehuq,
                                           levels = c(1, 2, 3, 4, 5),
                                           labels = c('Mobile home',
                                                      'Single-family detached house',
                                                      'Single-family attached house',
                                                      'Apartment in a building with 2 to 4 units',
                                                      'Apartment in a building with 5 or more units')),
                ownership_status = factor(kownrent,
                                          levels = c(1, 2, 3),
                                          labels = c('Owned',
                                                     'Rented',
                                                     'Occupied without payment of rent'))
)

In [23]:
names(df)

[1] "totaldol"   "totsqft_en" "kownrent"   "nhsldmem"   "typehuq"   
 [6] "fuelheat"   "hdd65"      "cdd65"      "ba_climate" "nweight"   
[11] "nweight1"   "nweight2"   "nweight3"   "nweight4"   "nweight5"  
[16] "nweight6"   "nweight7"   "nweight8"   "nweight9"   "nweight10" 
[21] "nweight11"  "nweight12"  "nweight13"  "nweight14"  "nweight15" 
[26] "nweight16"  "nweight17"  "nweight18"  "nweight19"  "nweight20" 
[31] "nweight21"  "nweight22"  "nweight23"  "nweight24"  "nweight25" 
[36] "nweight26"  "nweight27"  "nweight28"  "nweight29"  "nweight30" 
[41] "nweight31"  "nweight32"  "nweight33"  "nweight34"  "nweight35" 
[46] "nweight36"  "nweight37"  "nweight38"  "nweight39"  "nweight40" 
[51] "nweight41"  "nweight42"  "nweight43"  "nweight44"  "nweight45" 
[56] "nweight46"  "nweight47"  "nweight48"  "nweight49"  "nweight50" 
[61] "nweight51"  "nweight52"  "nweight53"  "nweight54"  "nweight55" 
[66] "nweight56"  "nweight57"  "nweight58"  "nweight59"  "nweight60"

In [24]:
head(design)

Call: update(design, main_heating_fuel = factor(fuelheat, levels = c(-2, 
    5, 1, 2, 3, 7, 99), labels = c("Not applicable", "Electricity", 
    "Natural gas from underground pipes", "Propane (bottled gas)", 
    "Fuel oil", "Wood or pellets", "Other")))
Unstratified cluster jacknife (JK1) with 60 replicates and MSE variances.

In [36]:
head(df)

,totaldol,totsqft_en,kownrent,nhsldmem,typehuq,fuelheat,hdd65,cdd65,ba_climate,nweight,⋯,nweight51,nweight52,nweight53,nweight54,nweight55,nweight56,nweight57,nweight58,nweight59,nweight60
,<dbl>,<int>,<fct>,<int>,<fct>,<fct>,<int>,<int>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2656.89,2100,1,2,2,Natural gas from underground pipes,3844,1679,Mixed-Dry,3284.104,⋯,3335.188,3240.132,3429.728,3294.084,3397.713,3292.610,0.000,3369.768,3358.163,3404.031
2,975.00,590,2,1,5,Electricity,3766,1458,Mixed-Humid,9007.387,⋯,8955.288,9000.296,9290.375,9199.326,8958.782,9232.597,9140.427,9306.997,9061.782,8957.915
3,522.65,900,2,1,5,Natural gas from underground pipes,3819,1696,Mixed-Dry,5669.002,⋯,5635.561,5944.330,5683.500,5735.631,5674.564,5660.854,5917.193,5571.015,5887.092,5837.846
4,2061.77,2100,1,2,2,Natural gas from underground pipes,2614,1718,Mixed-Humid,5294.239,⋯,5447.619,5344.453,5437.803,5377.898,5357.470,5421.025,5365.230,5402.057,5402.772,5350.860
5,1463.04,800,2,2,5,Natural gas from underground pipes,4219,1363,Mixed-Humid,9935.465,⋯,10017.094,9954.142,10050.961,10018.844,10309.728,10142.763,10176.828,10043.143,10247.911,10110.301
6,2335.08,4520,2,4,2,Natural gas from underground pipes,901,3558,Hot-Humid,7249.540,⋯,7379.108,7498.391,7417.977,7230.113,7516.936,7370.323,7383.275,7507.097,7335.022,7594.642


In [35]:
df$fuelheat=factor(df$fuelheat)
df$typehuq=factor(df$typehuq)
df$kownrent=factor(df$kownrent)